In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import KNNImputer
import os 
import random
from scipy.interpolate import splrep, BSpline

In [ ]:
# quantity_longest_interval = [{'file': 'treated cubic esmond data ap-rs 07-03-2023.csv', 'interval_length': 173}, {'file': 'treated bbr esmond data rs-go 07-07-2023.csv', 'interval_length': 126}, {'file': 'treated bbr esmond data ap-ba 07-03-2023.csv', 'interval_length': 123}, {'file': 'treated cubic esmond data ap-ce 07-08-2023.csv', 'interval_length': 121}, {'file': 'treated bbr esmond data go-es 07-08-2023.csv', 'interval_length': 118}, {'file': 'treated cubic esmond data ap-rn 07-03-2023.csv', 'interval_length': 110}, {'file': 'treated bbr esmond data ap-rs 07-03-2023.csv', 'interval_length': 107}, {'file': 'treated cubic esmond data rs-es 07-07-2023.csv', 'interval_length': 107}, {'file': 'treated bbr esmond data ac-pa 07-03-2023.csv', 'interval_length': 106}, {'file': 'treated bbr esmond data rs-ce 07-07-2023.csv', 'interval_length': 102}]
# quantity_longest_interval

In [ ]:
def outlier_removal(df, column):

    df[column] = df[column].replace(-1, np.nan)

    r = df[column].dropna().to_numpy()
    
    if r.size == 0:
        print("Coluna não contém valores suficientes para análise.")
        return df

    r_max = np.max(r) 
    r = r / r_max  

    perc_min = []
    p_min = np.linspace(0.1, 2, 20)
    for i in p_min:
        perc_min.append(np.percentile(r, i))
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)  
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    perc_max = []
    p_max = np.linspace(98, 100, 20)
    for i in p_max:
        perc_max.append(np.percentile(r, i))
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)  
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    r_filtered = np.where((r < thres_min) | (r > thres_max), np.NaN, r)

    r_filtered = r_filtered * r_max  

    df_filtered = df.copy()
    df_filtered.loc[~df[column].isna(), column] = r_filtered

    return df_filtered

def generate_missing_data(df, porcentagem):
    df_copy = df.copy()
    df_copy = outlier_removal(df, column='Throughput') # Removing outliers
    quantidade = (porcentagem * len(df_copy) / 100)
    indices_substituir = random.sample(df_copy.index.tolist(), round(quantidade))
    df_copy.loc[indices_substituir, 'Throughput'] = np.nan
    return df_copy

def linear_interpolation(df, original_df, missing_percentage, limit_direction='both', order=1, method='linear'):
    # Ensure the 'Timestamp' column is in datetime format and set it as the index
    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
        original_df['Timestamp'] = pd.to_datetime(original_df['Timestamp'], errors='coerce')
        
        df = df.set_index('Timestamp')
        original_df = original_df.set_index('Timestamp')

    # Check for any remaining issues with the index format
    if not isinstance(df.index, pd.DatetimeIndex):
        print("The index is not a DatetimeIndex. Please ensure the 'Timestamp' column is in proper datetime format.")
        return df  # Return the original DataFrame if the conversion fails
    
    df = df.sort_index()
    original_df = original_df.sort_index()

    # Identify missing indices
    missing_indices = df[df['Throughput'].isnull()].index

    # Apply interpolation
    df_imputed = df.interpolate(method=method, order=order, limit_direction=limit_direction)

    # Plot the interpolated data
    df_imputed['Throughput'].plot(style='.-', figsize=(12, 8), title='Throughput with Linear Interpolation')
    plt.scatter(missing_indices, df_imputed.loc[missing_indices, 'Throughput'], color='red')
    
    # Set plot labels
    plt.xlabel('Time')
    plt.ylabel('Throughput')
    plt.show()

    # Calculate RMSE for imputed values only
    rmse = (np.sqrt(((original_df['Throughput'] - df_imputed['Throughput']).dropna() ** 2).mean()) / 1000000)
    # print(f"RMSE for imputed values: {rmse}")

    result = {
        "Missing Percentage": missing_percentage,
        "Interpolation Method": method,
        "Order": order,
        "Limit Direction": limit_direction,
        "RMSE": rmse
    }
    return result, df_imputed

def plot_rmses_result(results_list):
    # Convert data into a DataFrame for easy manipulation
    df = pd.DataFrame(results_list)

    # Sort and extract unique values
    missing_percentages = sorted(df['Missing Percentage'].unique())
    methods = df['Interpolation Method'].unique()
    num_methods = len(methods)

    # Set up the figure
    fig, ax = plt.subplots(figsize=(12, 8))

    # X-axis positions for each group
    x = np.arange(len(missing_percentages))
    width = 0.15  # Width of each bar

    # Plot each interpolation method as a separate bar in each group
    for i, method in enumerate(methods):
        # Filter the DataFrame for the current interpolation method
        df_method = df[df['Interpolation Method'] == method]
        
        # RMSE values for the current method across all missing percentages
        rmse_values = [df_method[df_method['Missing Percentage'] == pct]['RMSE'].values[0] for pct in missing_percentages]
        
        # Plot the bars
        ax.bar(x + i * width, rmse_values, width, label=method)

    # Labeling and formatting
    ax.set_xlabel('Missing Percentage')
    ax.set_ylabel('RMSE')
    ax.set_title('RMSE by Missing Percentage and Interpolation Method')
    ax.set_xticks(x + width * (num_methods - 1) / 2)
    ax.set_xticklabels([f'{pct}%' for pct in missing_percentages])
    ax.legend(title='Interpolation Method')

    plt.show()


In [ ]:
#Using the longest interval among 07-07-2023 datasets
df = pd.read_csv("../datasets/throughput/longest interval/treated cubic esmond data ap-rs 07-03-2023_longest_interval.csv")

In [ ]:
df_missing10 = generate_missing_data(df, 10)
df_missing20 = generate_missing_data(df, 20)
df_missing30 = generate_missing_data(df, 30)

In [ ]:
dfs_missing = [df_missing10, df_missing20, df_missing30]
interpolation_results = []

Testing interpolation methods with different datasets (with diff data failures etc)

In [ ]:
missing_percentage = 10
for missing in dfs_missing:
    result, _ = linear_interpolation(missing, df, missing_percentage)
    interpolation_results.append(result)
    result_polynomial_order2, _ = linear_interpolation(missing, df, missing_percentage, order=2, method='polynomial')
    interpolation_results.append(result_polynomial_order2)
    result_polynomial_order3, _ = linear_interpolation(missing, df, missing_percentage, order=3, method='polynomial')
    interpolation_results.append(result_polynomial_order3)
    result_spline, _ = linear_interpolation(missing, df, missing_percentage, order=3, method='spline') # The order has to be minimum 3
    interpolation_results.append(result_spline)
    result_time, _ = linear_interpolation(missing, df, missing_percentage, method='time') 
    interpolation_results.append(result_time)
    missing_percentage = missing_percentage + 10

In [ ]:
interpolation_results

This graph compares each technique. After a lot of tests, linear interpolation is better

In [ ]:
plot_rmses_result(interpolation_results)

Trying to find a best k value for datasets (without success)

In [ ]:
def knn(df, original_df, k_value, missing_percentage):
    # Create a copy of the DataFrame to avoid modifying the original
    df_copy = df.copy()
    
    # Introduce missing values based on missing_percentage
    missing_count = int(missing_percentage / 100 * len(df_copy))
    missing_indices = np.random.choice(df_copy.index, missing_count, replace=False)
    df_copy.loc[missing_indices, 'Throughput'] = np.nan

    # Normalize the data for KNNImputer (if not already normalized)
    throughput = df_copy['Throughput'].values.reshape(-1, 1)
    
    # Apply KNN Imputer
    imputer = KNNImputer(n_neighbors=k_value)
    df_copy['Throughput'] = imputer.fit_transform(throughput)
    
    # Calculate RMSE
    rmse = (np.sqrt(((original_df['Throughput'] - df_copy['Throughput']).dropna() ** 2).mean()) / 1000000)
    result = {
        "Missing Percentage": missing_percentage,
        "N Neighbours": k_value,
        "RMSE": rmse
    }
    
    return result, df_copy


In [ ]:
dfs_missing = [df_missing10, df_missing20, df_missing30]
knn_results = []

In [ ]:
missing_percentage = 10
for missing in dfs_missing:
    for i in range(1, 100):
        result, _ = knn(missing, df, i, missing_percentage)
        knn_results.append(result)
    missing_percentage = missing_percentage + 10

In [ ]:
knn_results

In [ ]:
import numpy as np
from sklearn.impute import KNNImputer

def knn_with_multiple_iterations(df, original_df, k_value, missing_percentage, iterations=10):
    rmse_values = []
    
    for _ in range(iterations):
        # Create a copy of the DataFrame to avoid modifying the original
        df_copy = df.copy()
        
        # Introduce missing values based on missing_percentage
        missing_count = int(missing_percentage / 100 * len(df_copy))
        missing_indices = np.random.choice(df_copy.index, missing_count, replace=False)
        df_copy.loc[missing_indices, 'Throughput'] = np.nan

        # Normalize the data for KNNImputer
        throughput = df_copy['Throughput'].values.reshape(-1, 1)
        
        # Apply KNN Imputer
        imputer = KNNImputer(n_neighbors=k_value)
        df_copy['Throughput'] = imputer.fit_transform(throughput)
        
        # Calculate RMSE
        rmse = (np.sqrt(((original_df['Throughput'] - df_copy['Throughput']).dropna() ** 2).mean()) / 1000000)
        rmse_values.append(rmse)

    # Calculate the mean and standard deviation of RMSE across all iterations
    mean_rmse = np.mean(rmse_values)
    std_rmse = np.std(rmse_values)
    
    result = {
        "Missing Percentage": missing_percentage,
        "N Neighbours": k_value,
        "Mean RMSE": mean_rmse,
        "Std RMSE": std_rmse
    }
    
    return result

# Example usage
knn_results = []
for k in range(1, 101):
    result = knn_with_multiple_iterations(df_missing10, df, k, missing_percentage=10, iterations=10)
    knn_results.append(result)


In [ ]:
knn_results

In [ ]:
import numpy as np
from sklearn.impute import KNNImputer
import pandas as pd

def cross_validate_knn(df, k_values, missing_percentage=10, iterations=5):
    results = []
    for k in k_values:
        rmse_list = []
        for _ in range(iterations):
            # Criar uma cópia do DataFrame e introduzir faltantes artificialmente
            df_sample = df.copy()
            missing_count = int(missing_percentage / 100 * len(df_sample))
            missing_indices = np.random.choice(df_sample.index, missing_count, replace=False)
            df_sample.loc[missing_indices, 'Throughput'] = np.nan

            # Aplicar KNN com k vizinhos
            imputer = KNNImputer(n_neighbors=k)
            df_imputed = df_sample.copy()
            df_imputed['Throughput'] = imputer.fit_transform(df_sample[['Throughput']])
            
            # Calcular o RMSE usando os valores originais para validação
            rmse = np.sqrt(((df['Throughput'] - df_imputed['Throughput']).dropna() ** 2).mean())
            rmse_list.append(rmse)

        # Média do RMSE para o valor de k após várias iterações
        avg_rmse = np.mean(rmse_list)
        results.append({'k': k, 'RMSE': avg_rmse})

    # Converter resultados para DataFrame e retornar k com menor RMSE médio
    results_df = pd.DataFrame(results)
    best_k = results_df.loc[results_df['RMSE'].idxmin(), 'k']
    return best_k, results_df

# Exemplo de uso
k_values = range(1, 101)
best_k, results_df = cross_validate_knn(df, k_values, missing_percentage=10)
print(f"Best k for imputation: {best_k}")
print(results_df)


In [ ]:
from collections import Counter
import numpy as np
from sklearn.impute import KNNImputer
import pandas as pd

def cross_validate_knn(df, k_values, missing_percentage=10, iterations=5):
    results = []
    for k in k_values:
        rmse_list = []
        for _ in range(iterations):
            # Criar uma cópia do DataFrame e introduzir faltantes artificialmente
            df_sample = df.copy()
            missing_count = int(missing_percentage / 100 * len(df_sample))
            missing_indices = np.random.choice(df_sample.index, missing_count, replace=False)
            df_sample.loc[missing_indices, 'Throughput'] = np.nan

            # Aplicar KNN com k vizinhos
            imputer = KNNImputer(n_neighbors=k)
            df_imputed = df_sample.copy()
            df_imputed['Throughput'] = imputer.fit_transform(df_sample[['Throughput']])
            
            # Calcular o RMSE usando os valores originais para validação
            rmse = np.sqrt(((df['Throughput'] - df_imputed['Throughput']).dropna() ** 2).mean())
            rmse_list.append(rmse)

        # Média do RMSE para o valor de k após várias iterações
        avg_rmse = np.mean(rmse_list)
        results.append({'k': k, 'RMSE': avg_rmse})

    # Converter resultados para DataFrame e retornar k com menor RMSE médio
    results_df = pd.DataFrame(results)
    best_k = results_df.loc[results_df['RMSE'].idxmin(), 'k']
    return best_k

# Função para rodar múltiplas execuções e contar o melhor k
def find_most_frequent_best_k(df, k_values, missing_percentage=10, num_runs=20):
    best_ks = []
    for _ in range(num_runs):
        best_k = cross_validate_knn(df, k_values, missing_percentage)
        best_ks.append(best_k)
    
    # Contar a frequência de cada melhor k
    k_counter = Counter(best_ks)
    most_common_k = k_counter.most_common(1)[0][0]
    print(f"Most frequent best k: {most_common_k} (appeared {k_counter[most_common_k]} times out of {num_runs})")
    print("All k frequencies:", k_counter)
    return most_common_k, k_counter

# Exemplo de uso
k_values = range(1, 101)
most_common_k, k_frequencies = find_most_frequent_best_k(df, k_values, missing_percentage=10, num_runs=100)


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

def moving_average_imputation(df, window_size):
    # Aplicar média móvel usando a janela especificada
    df_copy = df.copy()
    df_copy['Throughput'] = df_copy['Throughput'].fillna(df_copy['Throughput'].rolling(window=window_size, min_periods=1).mean())
    return df_copy

def cross_validate_moving_average(df, window_sizes, missing_percentage=10, iterations=5):
    results = []
    for window in window_sizes:
        rmse_list = []
        for _ in range(iterations):
            # Criar uma cópia do DataFrame e introduzir faltantes artificialmente
            df_sample = df.copy()
            missing_count = int(missing_percentage / 100 * len(df_sample))
            missing_indices = np.random.choice(df_sample.index, missing_count, replace=False)
            df_sample.loc[missing_indices, 'Throughput'] = np.nan

            # Aplicar a imputação com média móvel
            df_imputed = moving_average_imputation(df_sample, window)
            
            # Calcular o RMSE entre os valores originais e os imputados
            rmse = np.sqrt(((df['Throughput'] - df_imputed['Throughput']).dropna() ** 2).mean())
            rmse_list.append(rmse)

        # Média do RMSE para o tamanho de janela após várias iterações
        avg_rmse = np.mean(rmse_list)
        results.append({'window_size': window, 'RMSE': avg_rmse})

    # Converter resultados para DataFrame e retornar o tamanho de janela com menor RMSE médio
    results_df = pd.DataFrame(results)
    best_window = results_df.loc[results_df['RMSE'].idxmin(), 'window_size']
    return best_window, results_df

# Função para rodar múltiplas execuções e contar o melhor tamanho de janela
def find_most_frequent_best_window(df, window_sizes, missing_percentage=10, num_runs=20):
    best_windows = []
    for _ in range(num_runs):
        best_window, _ = cross_validate_moving_average(df, window_sizes, missing_percentage)
        best_windows.append(best_window)
    
    # Contar a frequência de cada melhor tamanho de janela
    window_counter = Counter(best_windows)
    most_common_window = window_counter.most_common(1)[0][0]
    print(f"Most frequent best window size: {most_common_window} (appeared {window_counter[most_common_window]} times out of {num_runs})")
    print("All window size frequencies:", window_counter)
    return most_common_window, window_counter

# Exemplo de uso
window_sizes = range(2, 51)  # Defina o intervalo de tamanhos de janela
most_common_window, window_frequencies = find_most_frequent_best_window(df, window_sizes, missing_percentage=10, num_runs=100)


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

def moving_average_imputation(df, window_size):
    # Aplicar a média móvel com a janela especificada para imputação
    df_copy = df.copy()
    df_copy['Throughput'] = df_copy['Throughput'].fillna(df_copy['Throughput'].rolling(window=window_size, min_periods=1).mean())
    return df_copy

def cross_validate_moving_average(df, window_sizes, missing_percentage=10, iterations=5):
    results = []
    for window in window_sizes:
        rmse_list = []
        for _ in range(iterations):
            # Criar uma cópia do DataFrame e introduzir faltantes artificialmente
            df_sample = df.copy()
            missing_count = int(missing_percentage / 100 * len(df_sample))
            missing_indices = np.random.choice(df_sample.index, missing_count, replace=False)
            df_sample.loc[missing_indices, 'Throughput'] = np.nan

            # Aplicar a imputação com média móvel
            df_imputed = moving_average_imputation(df_sample, window)
            
            # Calcular o RMSE entre os valores originais e os imputados
            rmse = np.sqrt(((df['Throughput'] - df_imputed['Throughput']).dropna() ** 2).mean())
            rmse_list.append(rmse)

        # Média do RMSE para o tamanho da janela após várias iterações
        avg_rmse = np.mean(rmse_list)
        results.append({'window_size': window, 'RMSE': avg_rmse})

    # Converter resultados para DataFrame e retornar o tamanho da janela com menor RMSE médio
    results_df = pd.DataFrame(results)
    best_window = results_df.loc[results_df['RMSE'].idxmin(), 'window_size']
    return best_window, results_df

# Função para rodar múltiplas execuções e contar o melhor tamanho de janela
def find_most_frequent_best_window(df, window_sizes, missing_percentage=10, num_runs=20):
    best_windows = []
    for _ in range(num_runs):
        best_window, _ = cross_validate_moving_average(df, window_sizes, missing_percentage)
        best_windows.append(best_window)
    
    # Contar a frequência de cada melhor tamanho de janela
    window_counter = Counter(best_windows)
    most_common_window = window_counter.most_common(1)[0][0]
    print(f"Most frequent best window size: {most_common_window} (appeared {window_counter[most_common_window]} times out of {num_runs})")
    print("All window size frequencies:", window_counter)
    return most_common_window, window_counter

# Exemplo de uso
window_sizes = range(2, 51)  # Defina o intervalo de tamanhos de janela para testar
most_common_window, window_frequencies = find_most_frequent_best_window(df, window_sizes, missing_percentage=10, num_runs=100)
